In [1]:
import mlflow
from pycaret.classification import setup, create_model, finalize_model, save_model, predict_model
from sklearn.metrics import log_loss, f1_score

In [2]:
def evaluate_models(lr_model, dt_model, test_df, raw_score=True):
    """
    Recebe 2 modelos e a base de teste.
    Calcula métricas e salva best_model como pkl.
    """
    preds_lr = predict_model(lr_model, data=test_df, raw_score=raw_score)
    preds_dt = predict_model(dt_model, data=test_df, raw_score=raw_score)

    # A coluna de score é 'Score' quando raw_score=False e 'Score_1' quando raw_score=True
    score_col = "Score" if not raw_score else "Score_1"

    ll_lr = log_loss(preds_lr["shot_made_flag"], preds_lr[score_col])
    f1_lr = f1_score(preds_lr["shot_made_flag"], preds_lr["Label"])
    mlflow.log_metric("lr_log_loss", ll_lr)
    mlflow.log_metric("lr_f1", f1_lr)

    ll_dt = log_loss(preds_dt["shot_made_flag"], preds_dt[score_col])
    f1_dt = f1_score(preds_dt["shot_made_flag"], preds_dt["Label"])
    mlflow.log_metric("dt_log_loss", ll_dt)
    mlflow.log_metric("dt_f1", f1_dt)

    if ll_lr < ll_dt:
        best = lr_model
        chosen = "LogisticRegression"
    else:
        best = dt_model
        chosen = "DecisionTree"

    mlflow.log_param("chosen_model", chosen)

    # Salvar best_model via PyCaret + MLflow
    save_model(best, "best_model")
    mlflow.log_artifact("best_model.pkl")

    # Retornar o objeto Python do melhor modelo, que Kedro salvará num .pkl
    return best

In [3]:
import pandas as pd
import mlflow
from pycaret.classification import (
    setup, create_model, finalize_model, 
    predict_model, save_model
)
from sklearn.metrics import log_loss, f1_score


In [6]:
train_df = pd.read_parquet("../data/05_model_input/base_train.parquet")
test_df  = pd.read_parquet("../data/05_model_input/base_test.parquet")

train_df.shape, test_df.shape


((16228, 7), (4057, 7))

In [7]:
setup(
    data=train_df,
    target="shot_made_flag",
    session_id=42,
    verbose=False
)

lr = create_model("lr")      # cria o modelo
lr_final = finalize_model(lr) # finaliza


<IPython.core.display.HTML object>

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.5880,0.6183,0.5166,0.5761,0.5447,0.1706,0.1714
1,0.5907,0.6024,0.5092,0.5811,0.5428,0.1752,0.1764
2,0.5854,0.6107,0.4908,0.5770,0.5304,0.1636,0.1653
3,0.5836,0.6029,0.5037,0.5723,0.5358,0.1611,0.1622
4,0.6012,0.6230,0.5111,0.5957,0.5501,0.1958,0.1976
5,0.5607,0.5815,0.4871,0.5443,0.5141,0.1156,0.1162
6,0.5511,0.5791,0.4446,0.5356,0.4859,0.0935,0.0948
7,0.5801,0.6145,0.4659,0.5750,0.5148,0.1518,0.1544
8,0.5810,0.5988,0.4954,0.5711,0.5306,0.1556,0.1569


<IPython.core.display.HTML object>

In [8]:
preds_lr = predict_model(lr_final, data=test_df)
print(preds_lr.columns)
preds_lr.head()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Logistic Regression,0.5716,0.5945,0.4724,0.5610,0.5129,0.1356,0.1372


Index(['lat', 'lon', 'minutes_remaining', 'period', 'playoffs',
       'shot_distance', 'shot_made_flag', 'prediction_label',
       'prediction_score'],
      dtype='object')


,lat,lon,minutes_remaining,period,playoffs,shot_distance,shot_made_flag,prediction_label,prediction_score
30558,33.888302,-118.382797,11,3,1,19,0.0,0,0.5964
520,34.021301,-118.159798,2,3,0,11,0.0,0,0.5651
25613,33.983299,-118.120796,10,2,0,16,0.0,0,0.5997
15368,34.044300,-118.269798,0,4,0,0,1.0,1,0.5726
9175,34.044300,-118.269798,0,1,0,0,0.0,1,0.5943


In [10]:
ll_lr = log_loss(preds_lr["shot_made_flag"], preds_lr["prediction_score"])
f1_lr = f1_score(preds_lr["shot_made_flag"], preds_lr["prediction_label"])

print("Log Loss =", ll_lr)
print("F1 =", f1_lr)


Log Loss = 0.7159574603735508
F1 = 0.5128923766816144


In [11]:
preds_lr = predict_model(lr_final, data=test_df, raw_score=True)
print(preds_lr.columns)
preds_lr.head()


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Logistic Regression,0.5716,0.5945,0.4724,0.5610,0.5129,0.1356,0.1372


Index(['lat', 'lon', 'minutes_remaining', 'period', 'playoffs',
       'shot_distance', 'shot_made_flag', 'prediction_label',
       'prediction_score_0', 'prediction_score_1'],
      dtype='object')


,lat,lon,minutes_remaining,period,playoffs,shot_distance,shot_made_flag,prediction_label,prediction_score_0,prediction_score_1
30558,33.888302,-118.382797,11,3,1,19,0.0,0,0.5964,0.4036
520,34.021301,-118.159798,2,3,0,11,0.0,0,0.5651,0.4349
25613,33.983299,-118.120796,10,2,0,16,0.0,0,0.5997,0.4003
15368,34.044300,-118.269798,0,4,0,0,1.0,1,0.4274,0.5726
9175,34.044300,-118.269798,0,1,0,0,0.0,1,0.4057,0.5943


In [12]:
dt = create_model("dt")
dt_final = finalize_model(dt)

preds_dt = predict_model(dt_final, data=test_df)
print(preds_dt.columns)
preds_dt.head()


<IPython.core.display.HTML object>

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.5282,0.5106,0.5756,0.5049,0.5379,0.0601,0.0607
1,0.5255,0.5076,0.5461,0.5025,0.5234,0.0527,0.0528
2,0.5335,0.5056,0.6107,0.5092,0.5554,0.0730,0.0744
3,0.5176,0.5033,0.5554,0.4951,0.5235,0.0383,0.0386
4,0.5290,0.5151,0.5738,0.5057,0.5376,0.0617,0.0622
5,0.5229,0.5113,0.5683,0.5000,0.5320,0.0495,0.0499
6,0.5687,0.5643,0.6144,0.5423,0.5761,0.1405,0.1416
7,0.5352,0.5156,0.5709,0.5124,0.5401,0.0731,0.0735
8,0.5273,0.5118,0.5727,0.5049,0.5367,0.0581,0.0586


<IPython.core.display.HTML object>

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Decision Tree Classifier,0.5502,0.5276,0.5968,0.5255,0.5589,0.1037,0.1046


Index(['lat', 'lon', 'minutes_remaining', 'period', 'playoffs',
       'shot_distance', 'shot_made_flag', 'prediction_label',
       'prediction_score'],
      dtype='object')


,lat,lon,minutes_remaining,period,playoffs,shot_distance,shot_made_flag,prediction_label,prediction_score
30558,33.888302,-118.382797,11,3,1,19,0.0,0,1.0000
520,34.021301,-118.159798,2,3,0,11,0.0,0,1.0000
25613,33.983299,-118.120796,10,2,0,16,0.0,1,1.0000
15368,34.044300,-118.269798,0,4,0,0,1.0,1,0.6098
9175,34.044300,-118.269798,0,1,0,0,0.0,1,0.6163


In [14]:
ll_dt = log_loss(preds_dt["shot_made_flag"], preds_dt["prediction_score"])
f1_dt = f1_score(preds_dt["shot_made_flag"], preds_dt["prediction_label"])

print("Log Loss =", ll_dt)
print("F1 =", f1_dt)

Log Loss = 16.418254885050388
F1 = 0.5588590766255741
